# PS-P1: Quantum-Safe Secure Channel - end-to-end notebook
Qiskit Fall Fest 2026, Phase 1, Post-Quantum Cryptography track. Run from the repository root, top to bottom.

* The channel is **classical software**: ML-KEM / ML-DSA / HKDF / AES-256-GCM on ordinary CPUs. Platforms A/B/C/D are CPU+memory budgets.
* Processors A and B (quantum coupling maps) are built in section 7 for the Phase 1 qualification.
* Sections that need `liboqs-python` or `qiskit` skip themselves cleanly when those are missing.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 200, "display.max_columns", 40)
try:
    from IPython.display import Image, display
except ImportError:                       # plain-Python fallback
    def display(x): print(x)
    Image = lambda filename, **k: f"[figure] {filename}"

from src import problem, process, metrics, routing, visualization
print(problem.PROBLEM_ID, "-", problem.PROBLEM_TITLE)

PS-P1 - Quantum-Safe Secure Channel


## 1. Parameter sets and available backends
The kit asks for the exact parameter sets. Sets whose library is missing are listed as skipped (install liboqs-python for ML-KEM/ML-DSA).

In [ ]:
for k, ps in problem.PARAMETER_SETS.items():
    print(f"{k:15s} {ps.describe()}")
AVAILABLE, SKIPPED = metrics.available_sets(list(problem.PARAMETER_SETS))
print("\navailable:", AVAILABLE)
print("skipped  :", {k: v[:60] + '...' for k, v in SKIPPED.items()})

## 2. Working demo: handshake, message and file transfer over localhost TCP

In [ ]:
payloads = problem.locate_payloads()
for k, p in payloads.items():
    print(k, p.size, "bytes", "|", p.source)

for key in AVAILABLE:
    out = process.tcp_demo(key, b"hello from the quantum-safe channel", file=payloads["100KB"].path, chunk_bytes=65536)
    print("\n==", key, "==")
    for name, v in out.items():
        print(f"  {name}: {v}")
display(Image(filename=str(ROOT / "results" / "figures" / "handshake_flow.png")) if (ROOT / "results/figures/handshake_flow.png").exists() else "run section 5 to draw figures")

## 3. Man-in-the-middle scenarios
An on-path relay swaps KEM keys, corrupts ciphertexts, forges the server hello, flips bits, replays and truncates. Every attack must be detected.

In [ ]:
mitm = metrics.run_mitm_table(AVAILABLE)
mitm[["param_set", "scenario", "expected_defence", "attack_detected", "pass"]]

## 4. Benchmark on Platforms A/B/C/D
`QUICK = True` is a smoke run (few repetitions). Set `QUICK = False` for the real numbers (about a minute per parameter set).

In [ ]:
QUICK = False
metrics.run_all(quick=QUICK, sets=AVAILABLE)

## 5. Results: laptop (A) vs edge (C), sizes, memory

In [ ]:
T = ROOT / "results" / "tables"
head = T / "headline_comparison.csv"
display(pd.read_csv(head).round(3) if head.exists() else "headline needs platforms A and C")
display(pd.read_csv(T / "primitive_sizes_and_ops.csv").round(3))
display(pd.read_csv(T / "predicted_handshake_bytes.csv"))   # predicted from FIPS 203/204 sizes, not measured

In [ ]:
made = visualization.make_all()
print(made)
for name in ["headline_laptop_vs_edge.png", "sizes_comparison.png", "memory_by_platform.png"]:
    f = ROOT / "results" / "figures" / name
    if f.exists():
        display(Image(filename=str(f)))

In [ ]:
bench = pd.read_csv(T / "pqc_benchmark.csv")
bench[["platform", "backend", "instance_id", "chunk_bytes", "handshake_ms_median", "transfer_ms_median", "tracemalloc_peak_kb",
       "within_mem_budget", "bytes_handshake", "record_overhead_pct", "handshake_correct", "payload_integrity_ok"]].round(3)

## 6. Platform D design
Record size chosen by a rule fixed in advance: the smallest record size reaching 90% of the best in-budget throughput.

In [ ]:
display(pd.read_csv(T / "chunk_sweep.csv").round(3))
print(json.dumps(json.loads((T / "chunk_sweep_recommendation.json").read_text()), indent=1))
display(Image(filename=str(ROOT / "results" / "figures" / "chunk_sweep.png")))

## 7. Processors A and B (qualification)
Built from scratch as participant custom architectures (Ring & 3x3 Grid); candidate topologies explored for routing overhead. Processor C is study-only.

In [ ]:
print(routing.write_processor_files())          # regenerate JSON from the kit spec and validate
for key in ("A", "B"):
    proc = routing.load_processor(key)
    print(proc["name"], proc["num_qubits"], "qubits", proc["coupling_map"])
display(routing.processor_graph_table())
display(routing.routing_estimate_table())
for name in ("processor_A_graph.png", "processor_B_graph.png"):
    display(Image(filename=str(ROOT / "results" / "figures" / name)))

## 8. Optional quantum experiment: the same circuit on Processor A and B (needs `qiskit`, `qiskit-aer`)
Grover search is why the channel uses AES-256. Required benchmark settings: basis {rz, sx, x, cx}, `optimization_level=1`, SABRE layout and routing, seed 42,
4096 shots, fixed noise model (0.001 / 0.005 / 0.01). This cell was **not executed** where the repo was authored.

In [ ]:
try:
    qc = routing.grover_circuit("101", iterations=2)
    ab = routing.run_ab_experiment(qc, expected="101", shots=4096)
    ab.to_csv(T / "ab_comparison.csv", index=False)
    display(ab)
except ImportError as exc:
    print("qiskit / qiskit-aer not installed - skipping the A/B quantum experiment:", exc)

## 9. Closing checklist (Challenge Kit section 13)
- [ ] one track and one problem statement (PS-P1) - [ ] official payloads used (copy them to `data/raw/`) - [ ] reference documented (ECDH P-256 baseline)
- [ ] implementation runs end to end - [ ] Platforms A/B/C measured, Platform D designed and re-run - [ ] metrics reported: time, memory, code size, bytes exchanged
- [ ] README, report, requirements, results present - [ ] another person can reproduce from the README